In [1]:
import pandas as pd

In [2]:
X_train = pd.read_csv('data/X_train.csv')
X_valid = pd.read_csv('data/X_valid.csv')
y_train = pd.read_csv('data/Y_train.csv')
y_valid = pd.read_csv('data/Y_valid.csv')
X_train.head()


,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,windspeed,day_repaired_sin,...,avg3_windspeed_minus_cloud,avg3_cloud_windspeed_times,avg3_cloud_div_windspeed,avg3_windspeed_div_cloud,avg3_sunshine_windspeed_plus,avg3_sunshine_minus_windspeed,avg3_windspeed_minus_sunshine,avg3_sunshine_windspeed_times,avg3_sunshine_div_windspeed,avg3_windspeed_div_sunshine
0,1010.8,24.3,21.4,20.4,19.5,86.0,77.0,7.3,9.5,0.985139,...,-60.7,2464.7,3.4,0.3,28.8,-27.2,27.2,23.6,0.0,127.2
1,1012.0,26.5,24.4,23.3,22.2,90.0,88.0,0.6,37.5,0.908324,...,-70.8,1197.9,6.4,0.2,17.4,-11.1,11.1,53.0,0.2,14.2
2,1013.5,28.9,26.5,24.8,20.6,76.0,54.0,4.5,16.7,-0.970942,...,-50.6,2410.5,2.8,0.4,32.4,-27.1,27.1,82.8,0.1,11.6
3,1013.9,20.9,20.6,19.4,19.9,73.0,83.0,2.0,14.8,0.495009,...,-48.8,1207.3,4.0,0.6,23.7,-16.0,16.0,94.4,0.2,3.4
4,1013.4,27.8,24.4,23.1,22.1,87.0,88.0,1.6,13.0,0.974928,...,-69.2,1250.3,6.6,0.2,15.1,-14.5,14.5,4.4,0.0,55.1


# Data preparation

**Normalization**

In [3]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
data_standardScaler = scaler.fit_transform(X_train)
X_train_standardScaler = pd.DataFrame(data_standardScaler, columns=X_train.columns)
X_val_standardScaler = pd.DataFrame(scaler.fit_transform(X_valid), columns=X_valid.columns)

# CatBoost

In [4]:
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score

### Not Normalized

In [5]:
categorical_features_indices = [i for i, col in enumerate(X_train.columns) if X_train[col].dtype == 'object']

model = CatBoostClassifier(
    iterations=400,     
    learning_rate=0.05,    
    depth=6,           
    cat_features=categorical_features_indices, 
    verbose=50        
)

model.fit(X_train, y_train)

y_pred = model.predict(X_valid)

accuracy = accuracy_score(y_valid, y_pred)
print("Accuracy:", accuracy)

0:	learn: 0.6531523	total: 252ms	remaining: 1m 40s
50:	learn: 0.2653886	total: 7.17s	remaining: 49.1s
100:	learn: 0.2140566	total: 12.2s	remaining: 36.3s
150:	learn: 0.1759147	total: 17s	remaining: 28s
200:	learn: 0.1427263	total: 22.6s	remaining: 22.4s
250:	learn: 0.1145730	total: 27.3s	remaining: 16.2s
300:	learn: 0.0914209	total: 31.6s	remaining: 10.4s
350:	learn: 0.0735559	total: 35.9s	remaining: 5.01s
399:	learn: 0.0605109	total: 40.2s	remaining: 0us
Accuracy: 0.8515981735159818


### Standarized

In [7]:
categorical_features_indices = [i for i, col in enumerate(X_train_standardScaler.columns) if X_train_standardScaler[col].dtype == 'object']

model = CatBoostClassifier(
    iterations=400,     
    learning_rate=0.05,   
    depth=6,       
    cat_features=categorical_features_indices,
    verbose=50      
)

model.fit(X_train_standardScaler, y_train)

y_pred = model.predict(X_val_standardScaler)

accuracy = accuracy_score(y_valid, y_pred)
print("Accuracy:", accuracy)

0:	learn: 0.6525748	total: 84.1ms	remaining: 33.6s
50:	learn: 0.2694576	total: 5.62s	remaining: 38.4s
100:	learn: 0.2210207	total: 11s	remaining: 32.5s
150:	learn: 0.1813111	total: 14.9s	remaining: 24.5s
200:	learn: 0.1491341	total: 18.7s	remaining: 18.5s
250:	learn: 0.1171543	total: 22.5s	remaining: 13.4s
300:	learn: 0.0944576	total: 26.3s	remaining: 8.66s
350:	learn: 0.0751840	total: 30.1s	remaining: 4.21s
399:	learn: 0.0618707	total: 34.1s	remaining: 0us
Accuracy: 0.867579908675799


**early stopping**

In [8]:
categorical_features_indices = [i for i, col in enumerate(X_train_standardScaler.columns) if X_train_standardScaler[col].dtype == 'object']

model = CatBoostClassifier(
    iterations=400,      
    learning_rate=0.01,    
    depth=6,               
    cat_features=categorical_features_indices, 
    early_stopping_rounds=50,  
    verbose=50,            
    random_seed=42       
)

model.fit(
    X_train_standardScaler, y_train,
    eval_set=(X_val_standardScaler, y_valid)
)

y_val_pred = model.predict(X_val_standardScaler)
accuracy_val = accuracy_score(y_valid, y_val_pred)
print("Validation Accuracy:", accuracy_val)

0:	learn: 0.6851613	test: 0.6849352	best: 0.6849352 (0)	total: 112ms	remaining: 44.6s
50:	learn: 0.4376182	test: 0.4530903	best: 0.4530903 (50)	total: 6.31s	remaining: 43.2s
100:	learn: 0.3495961	test: 0.3824970	best: 0.3824970 (100)	total: 10.9s	remaining: 32.3s
150:	learn: 0.3098120	test: 0.3567279	best: 0.3567279 (150)	total: 16s	remaining: 26.4s
200:	learn: 0.2853186	test: 0.3463416	best: 0.3463416 (200)	total: 20.3s	remaining: 20.1s
250:	learn: 0.2684122	test: 0.3425845	best: 0.3425845 (250)	total: 24.5s	remaining: 14.5s
300:	learn: 0.2548499	test: 0.3419933	best: 0.3418446 (275)	total: 28.9s	remaining: 9.52s
350:	learn: 0.2451268	test: 0.3411760	best: 0.3410600 (345)	total: 33s	remaining: 4.61s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.3410599918
bestIteration = 345

Shrink model to first 346 iterations.
Validation Accuracy: 0.8721461187214612


In [9]:
categorical_features_indices = [i for i, col in enumerate(X_train_standardScaler.columns) if X_train_standardScaler[col].dtype == 'object']

model = CatBoostClassifier(
    iterations=1000,     
    learning_rate=0.005,   
    depth=6,           
    cat_features=categorical_features_indices, 
    early_stopping_rounds=50, 
    verbose=50,             
    random_seed=42    
)

model.fit(
    X_train_standardScaler, y_train,
    eval_set=(X_val_standardScaler, y_valid)
)

y_val_pred = model.predict(X_val_standardScaler)
accuracy_val = accuracy_score(y_valid, y_val_pred)
print("Validation Accuracy:", accuracy_val)

0:	learn: 0.6891417	test: 0.6890255	best: 0.6890255 (0)	total: 114ms	remaining: 1m 54s
50:	learn: 0.5297496	test: 0.5376651	best: 0.5376651 (50)	total: 5.67s	remaining: 1m 45s
100:	learn: 0.4380312	test: 0.4578055	best: 0.4578055 (100)	total: 10.7s	remaining: 1m 35s
150:	learn: 0.3848296	test: 0.4130687	best: 0.4130687 (150)	total: 15.9s	remaining: 1m 29s
200:	learn: 0.3521663	test: 0.3853051	best: 0.3853051 (200)	total: 21.2s	remaining: 1m 24s
250:	learn: 0.3292092	test: 0.3683577	best: 0.3683577 (250)	total: 26.3s	remaining: 1m 18s
300:	learn: 0.3117689	test: 0.3586535	best: 0.3586535 (300)	total: 31.6s	remaining: 1m 13s
350:	learn: 0.2978062	test: 0.3517946	best: 0.3517946 (350)	total: 36.9s	remaining: 1m 8s
400:	learn: 0.2867104	test: 0.3481551	best: 0.3481551 (400)	total: 42s	remaining: 1m 2s
450:	learn: 0.2775068	test: 0.3460705	best: 0.3460547 (448)	total: 47.1s	remaining: 57.3s
500:	learn: 0.2690905	test: 0.3441433	best: 0.3441433 (500)	total: 52.3s	remaining: 52.1s
550:	learn:

In [10]:
categorical_features_indices = [i for i, col in enumerate(X_train_standardScaler.columns) if X_train_standardScaler[col].dtype == 'object']

model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.005,      
    depth=10,     
    cat_features=categorical_features_indices,
    early_stopping_rounds=50, 
    verbose=50,    
    random_seed=42 
)

model.fit(
    X_train_standardScaler, y_train,
    eval_set=(X_val_standardScaler, y_valid)
)

y_val_pred = model.predict(X_val_standardScaler)
accuracy_val = accuracy_score(y_valid, y_val_pred)
print("Validation Accuracy:", accuracy_val)

0:	learn: 0.6887756	test: 0.6892968	best: 0.6892968 (0)	total: 2.8s	remaining: 46m 37s
50:	learn: 0.5145374	test: 0.5388482	best: 0.5388482 (50)	total: 2m 20s	remaining: 43m 27s
100:	learn: 0.4131456	test: 0.4547983	best: 0.4547983 (100)	total: 4m 8s	remaining: 36m 54s
150:	learn: 0.3474402	test: 0.4075385	best: 0.4075385 (150)	total: 6m 6s	remaining: 34m 18s
200:	learn: 0.3031416	test: 0.3805076	best: 0.3805076 (200)	total: 7m 55s	remaining: 31m 30s
250:	learn: 0.2713540	test: 0.3649214	best: 0.3649214 (250)	total: 9m 43s	remaining: 29m 1s
300:	learn: 0.2459435	test: 0.3551700	best: 0.3551700 (300)	total: 11m 36s	remaining: 26m 57s
350:	learn: 0.2245424	test: 0.3492664	best: 0.3492664 (350)	total: 13m 26s	remaining: 24m 50s
400:	learn: 0.2066531	test: 0.3448467	best: 0.3448467 (400)	total: 15m 14s	remaining: 22m 46s
450:	learn: 0.1911078	test: 0.3416896	best: 0.3416552 (449)	total: 17m 5s	remaining: 20m 47s
500:	learn: 0.1782652	test: 0.3396209	best: 0.3396209 (500)	total: 19m 19s	rem